# Can FinBERT improve analyst triage of real SEC earnings releases?

This worked final-project example uses **real SEC evidence**, a **pinned FinBERT
checkpoint**, a **transparent baseline**, an **untouched later period**, and a
specific decision: which one third of releases receive scarce analyst review.

The notebook uses committed model scores by default. The accompanying downloader
and `analysis.py --score-finbert` recreate the primary-source and model pipeline.

This notebook is **Pathway 1**: the prose, code, results, limitations, and
references form one integrated report. Pathway 2 uses a shorter analysis notebook
plus a separately rendered Quarto paper.


## 1. Start with a question that can lose

> Does pinned FinBERT rank next-quarter year-over-year revenue growth better than
> a transparent word-count lexicon on 2024–2025 releases from a fixed six-issuer
> cohort?

FinBERT can lose by tying the baseline, reversing in the later period, or producing
an improvement too uncertain to justify replacing the cheap reader.


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "analysis.py").exists():
    candidates = [
        ROOT.parent,
        ROOT / "project" / "example_real_sec",
        Path("/content/uc_finmlai/project/example_real_sec"),
        Path("/content/aiinfinance/project/example_real_sec"),
    ]
    ROOT = next((path for path in candidates if (path / "analysis.py").exists()), None)
    if ROOT is None:
        raise FileNotFoundError(
            "Open this notebook from a clone of smc77/uc_finmlai, or run it "
            "inside project/example_real_sec so its compact data and figures are available."
        )
sys.path.insert(0, str(ROOT))
import analysis as A

frame = pd.read_csv(ROOT / "data/derived/scored_events.csv", dtype={"cik": str})
summary = json.loads((ROOT / "data/derived/summary.json").read_text())
print(f"{len(frame)} observations: {(frame.split == 'development').sum()} development, {(frame.split == 'test').sum()} test")


## 2. Understand the source before calculating anything

### What is an 8-K?

A **Form 8-K** is a company's current report for specified events. It is not the
regular quarterly 10-Q or annual 10-K. In our selected filings, **Item 2.02** says
the report concerns results of operations and financial condition; **Item 9.01**
lists the exhibits; and **Exhibit 99.1** is the attached earnings release we score.

Open the exact Apple example:

- [SEC filing-detail page](https://www.sec.gov/Archives/edgar/data/320193/000032019323000005/0000320193-23-000005-index.html)
- [selected Exhibit 99.1](https://www.sec.gov/Archives/edgar/data/320193/000032019323000005/a8-kex991q1202312312022.htm)

The filing page is the folder and index. The exhibit is the document the model
reads. The future outcome comes from structured XBRL facts in a later filing.


In [ ]:
display(Image(filename=str(ROOT / "figures/05_sec_filing_page.png")))
display(Image(filename=str(ROOT / "figures/06_sec_exhibit.png")))


### Inspect the corresponding research record

The feature is the selected EX-99 earnings release. The record also retains the
8-K accession, SEC URL, acceptance time, exhibit hash, future label date, and future
XBRL accession.


In [ ]:
columns = ["ticker", "accession_number", "decision_time_utc", "release_type",
           "exhibit_url", "release_sha256", "outcome_quarter_end", "outcome_available_date",
           "outcome_accession_number", "split"]
apple = frame.loc[frame.accession_number == "0000320193-23-000005", columns].iloc[0]
display(apple.to_frame("recorded value"))


### Pause 1 — Which information may the score use?

Before revealing the timeline, write down:

1. When does the release text first become admissible?
2. Which date belongs to the future outcome?
3. Why would aligning the text to quarter end be wrong?


In [ ]:
display(Image(filename=str(ROOT / "figures/01_information_clock.png")))


**Reveal.** The 8-K acceptance is the earliest-public-time bound for the text. The
next-quarter revenue label is not available until a later filing. Quarter end describes
the business period; it is not the publication time.


## 3. Make the population and test boundary visible

The fixed cohort contains six large issuers. The first 45 observations are
development evidence. The 23 observations accepted during 2024–2025 form the later
test. This is a deliberately small teaching cohort, not a representative market sample.


In [ ]:
display(Image(filename=str(ROOT / "figures/02_cohort.png")))


## 4. Compare one cheap reader with one pinned candidate

- **Baseline:** the printed finance-word vocabulary in `analysis.py`.
- **Candidate:** `ProsusAI/finbert` at revision
  `0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031`.
- **Long documents:** average class probabilities across at most eight evenly spaced
  480-token windows; score = positive probability − negative probability.

The checkpoint is not tuned on these releases, and the later test does not select a
different window rule.


### Pause 2 — Would these two point estimates be enough?

Reveal the point-estimate chart. Before moving on, decide whether a test IC of 0.62
versus 0.50 would justify replacing the transparent baseline. Write down the additional
number you need.


In [ ]:
display(Image(filename=str(ROOT / "figures/03a_point_estimates.png")))


## 5. Uncertainty changes the sentence

The chart below resamples issuers rather than pretending 23 releases are independent.
Both individual intervals are wide. More importantly, the paired 90% interval for
FinBERT minus lexicon is **−0.35 to +0.55**.


In [ ]:
display(Image(filename=str(ROOT / "figures/03_model_comparison.png")))
print("Paired IC difference:", round(summary["test"]["finbert_minus_lexicon_ic"], 3))
print("90% interval:", [round(x, 3) for x in summary["test"]["finbert_minus_lexicon_ic_90"]])


### Pause 3 — Convert the score into a decision

Assume analyst capacity is fixed at one third of releases. Predict what the next plot
must show before it can support that policy. Is mean selected revenue growth enough, or
must the figure retain the dispersion and the unselected observations too?


In [ ]:
display(Image(filename=str(ROOT / "figures/04_review_policy.png")))


## 6. The warranted conclusion

FinBERT's test point estimate is higher, and its top third contains high-growth future
quarters. But the same is true for the lexicon, the paired ranking interval includes a
large reversal, and NVIDIA contributes unusually large growth observations.

**Conclusion:** In this fixed six-issuer test, pinned FinBERT does not establish a
stable incremental ranking advantage over the transparent baseline. Keep the baseline
and require a broader issuer population plus a genuinely new later test before changing
the review policy.

That is a useful final-project result. The method was allowed to lose, the result is
visible, and the decision authority matches the evidence.


## 7. Reproducibility and limits

- `download_sec_data.py` recreates the SEC event/outcome ledger and hashes its sources.
- `analysis.py --score-finbert` recreates the scores from the pinned checkpoint.
- `summary.json` contains every number quoted above.
- `RESEARCH_RECORD.md` distinguishes the clock, procedure, result, and limitations.

This is historical teaching evidence, not a preregistered study, a market-wide sample,
or a trading backtest. It estimates neither returns nor implementation costs.
